EVT Fitting and decision boundary calculation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import genpareto

# ==========================================
# 1. LOAD CALIBRATED DATA
# ==========================================
print("="*40)
print("FINAL EVT CALIBRATION")
print("="*40)

# Ensuring we use the correct filename from your previous success
FILE_NAME = "inference_results" 

try:
    data = np.load(FILE_NAME)
    max_peaks = data['max_peaks']
    # Filter out any non-finite values just in case
    max_peaks = max_peaks[np.isfinite(max_peaks)]
    
    g_mean = data['calibration_mean']
    g_std = data['calibration_std']
    
    print(f"Loaded {len(max_peaks)} light curve maxima.")
    print(f"Calibration Constants: Mean={g_mean:.4f}, Std={g_std:.4f}")
except Exception as e:
    print(f"Error loading data: {e}")
    exit()

# ==========================================
# 2. DEFINE THE THRESHOLD (u) FOR TAIL FITTING
# ==========================================
u = np.percentile(max_peaks, 95)
excesses = max_peaks[max_peaks > u] - u

print(f"Tail fitting threshold (u): {u:.4f}")
print(f"Number of points in tail: {len(excesses)}")

# ==========================================
# 3. FIT THE GENERALIZED PARETO DISTRIBUTION
# ==========================================
# We fix floc=0 because 'excesses' start at 0 by definition.
# This prevents the ValueError: zero-size array or optimization failure.
shape, loc, scale = genpareto.fit(excesses, floc=0)

print("-" * 30)
print("GPD FIT PARAMETERS")
print("-" * 30)
print(f"Shape (xi):  {shape:.4f}")
print(f"Scale (sig): {scale:.4f}")

# ==========================================
# 4. CALCULATE DETECTION BOUNDARY
# ==========================================
p_target = 0.01 
N = len(max_peaks)
Nu = len(excesses)

if shape != 0:
    detection_threshold = u + (scale / shape) * (((N * p_target) / Nu)**(-shape) - 1)
else:
    detection_threshold = u - scale * np.log((N * p_target) / Nu)

print("-" * 30)
print(f"FINAL FLARE DETECTION BOUNDARY: {detection_threshold:.4f} Sigma")
print("-" * 40)

# ==========================================
# 5. DIAGNOSTIC PLOTS
# ==========================================
plt.figure(figsize=(12, 5))

# Plot A: Tail Distribution and Fit
plt.subplot(1, 2, 1)
plt.hist(excesses, bins=30, density=True, alpha=0.6, color='teal', label='Empirical Excesses')
x = np.linspace(0, excesses.max(), 100)
pdf = genpareto.pdf(x, shape, loc, scale)
plt.plot(x, pdf, 'r-', lw=2, label='GPD Fit')
plt.title("GPD Tail Fit")
plt.xlabel("Excess over Threshold (u)")
plt.ylabel("Probability Density")
plt.legend()

# Plot B: Global Distribution with Final Boundary
plt.subplot(1, 2, 2)
plt.hist(max_peaks, bins=100, color='darkblue', alpha=0.7, log=True)
plt.axvline(detection_threshold, color='red', linestyle='--', label=f'Threshold ({detection_threshold:.2f})')
plt.title("Final Detection Boundary")
plt.xlabel("Calibrated Max Z-Score")
plt.ylabel("Frequency (Log Scale)")
plt.legend()

plt.tight_layout()
plt.savefig("final_evt_calibration_v5.png")
plt.show()

# Save final parameters
np.savez("final_flare_detector_config.npz", 
         g_mean=g_mean, 
         g_std=g_std, 
         detection_threshold=detection_threshold,
         u=u, shape=shape, scale=scale)

Decision Boundary Stability

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import genpareto

# ==========================================
# 1. LOAD CALIBRATED DATA
# ==========================================
FILE_NAME = "inference_results_calibrated_v5.npz"

data = np.load(FILE_NAME)
max_peaks = data["max_peaks"]
max_peaks = max_peaks[np.isfinite(max_peaks)]

print(f"Loaded {len(max_peaks)} maxima.")

# ==========================================
# 2. SETTINGS
# ==========================================
p_target = 0.01   # false alarm rate
N = len(max_peaks)

# thresholds to test
percentiles = np.arange(90, 99, 1)

# storage
u_list = []
xi_list = []
scale_list = []
det_thresh_list = []
Nu_list = []

# ==========================================
# 3. LOOP OVER THRESHOLDS
# ==========================================
print("\nRunning threshold stability test...")

for pct in percentiles:

    u = np.percentile(max_peaks, pct)
    excesses = max_peaks[max_peaks > u] - u
    Nu = len(excesses)

    # skip if too few points
    if Nu < 30:
        print(f"Skipping {pct}% (too few tail points)")
        continue

    # GPD fit
    try:
        xi, loc, scale = genpareto.fit(excesses, floc=0)
    except:
        continue

    # EVT detection threshold
    if xi != 0:
        det = u + (scale/xi) * (((N*p_target)/Nu)**(-xi) - 1)
    else:
        det = u - scale*np.log((N*p_target)/Nu)

    # store
    u_list.append(u)
    xi_list.append(xi)
    scale_list.append(scale)
    det_thresh_list.append(det)
    Nu_list.append(Nu)

    print(f"{pct}% | u={u:.3f} | xi={xi:.3f} | det={det:.2f} | Nu={Nu}")

# ==========================================
# 4. STABILITY PLOTS
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(15,4))

# --- Shape parameter stability ---
axes[0].plot(percentiles[:len(xi_list)], xi_list, marker='o')
axes[0].axhline(np.mean(xi_list), color='r', ls='--', alpha=0.5)
axes[0].set_title("Shape Parameter (ξ) vs Threshold")
axes[0].set_xlabel("Threshold Percentile")
axes[0].set_ylabel("ξ")

# --- Detection boundary stability ---
axes[1].plot(percentiles[:len(det_thresh_list)],
             det_thresh_list, marker='o')
axes[1].set_title("Detection Boundary Stability")
axes[1].set_xlabel("Threshold Percentile")
axes[1].set_ylabel("Boundary (Sigma)")

# --- Tail sample size ---
axes[2].plot(percentiles[:len(Nu_list)], Nu_list, marker='o')
axes[2].set_title("Tail Sample Size")
axes[2].set_xlabel("Threshold Percentile")
axes[2].set_ylabel("Number of Excesses")

plt.tight_layout()
plt.savefig("evt_threshold_stability.png")
plt.show()

# ==========================================
# 5. SUMMARY
# ==========================================
print("\n==== STABILITY SUMMARY ====")
print(f"Mean xi: {np.mean(xi_list):.4f}")
print(f"Std xi : {np.std(xi_list):.4f}")
print(f"Detection boundary range: "
      f"{np.min(det_thresh_list):.2f} — {np.max(det_thresh_list):.2f}")
